# Análisis ConnectaTel

Como **analista de datos**, tu objetivo es evaluar el **comportamiento de los clientes** de una empresa de telecomunicaciones en Latinoamérica, ConnectaTel. 

Trabajaremos con información registrada **hasta el año 2024**, lo cual permitirá analizar el comportamiento del negocio dentro de ese periodo.

Para ello trabajarás con tres datasets:  

- **plans.csv** → información de los planes actuales (precio, minutos incluidos, GB incluidos, costo por extra)  
- **users.csv** → información de los clientes (edad, ciudad, fecha de registro, plan, churn)  
- **usage.csv** → detalle del **uso real** de los servicios (llamadas y mensajes)  

Deberás **explorar**, **limpiar** y **analizar** estos datos para construir un **perfil estadístico** de los clientes, detectar **comportamientos atípicos** y crear **segmentos de clientes**.  

Este análisis te permitirá **identificar patrones de consumo**, **diseñar estrategias de retención** y **sugerir mejoras en los planes** ofrecidos por la empresa.

> 💡 Antes de empezar, recuerda pensar de forma **programática**: ¿qué pasos necesitas? ¿En qué orden? ¿Qué quieres medir y por qué?


# --- 
## 🧩 Paso 1: Cargar y explorar

Antes de limpiar o combinar los datos, es necesario **familiarizarte con la estructura de los tres datasets**.  
En esta etapa, validarás que los archivos se carguen correctamente, conocerás sus columnas y tipos de datos, y detectarás posibles inconsistencias.

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:**  
Tener los **3 datasets listos en memoria**, entender su contenido y realizar una revisión preliminar.

**Instrucciones:**  
- Importa las librerías necesarias (por ejemplo `pandas`, `seaborn`, `matplotlib.pyplot`)
- Carga los archivos CSV usando `pd.read_csv()`:
  - **`/datasets/plans.csv`**  
  - **`/datasets/users_latam.csv`**  
  - **`/datasets/usage.csv`**  
- Guarda los DataFrames en las variables: `plans`, `users`, `usage`.  
- Muestra las primeras filas de cada DataFrame usando `.head()`.


# import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# plans = pd.read_csv('/datasets/plans.csv')
users = pd.read_csv('/datasets/users_latam.csv')
usage = pd.read_csv('/datasets/usage.csv')

# plans.head()

# users.head()

# usage.head()

# **Tip:** Si no usas `print()` la tabla se vera mejor.

# print("plans", plans.shape)
print("users", users.shape)
print("usage", usage.shape)

# plans.info()

# users.info()

# usage.info()

# usage.info()

# ---

## 🧩Paso 2: Identificación de problemas de calidad de datos

### 2.1 Revisión de valores nulos

**🎯 Objetivo:**  
Detectar la presencia y magnitud de valores faltantes para evaluar si afectan el análisis o requieren imputación/eliminación.

**Instrucciones:**  
- Cuenta valores nulos por columna para cada dataset.
- Calcula la proporción de nulos por columna para cada dataset.

El dataset `plans` solamente tiene 2 renglones y se puede observar que no tiene ausentes, por ello no necesita exploración adicional.

<br>
<details>
<summary>Haz clic para ver la pista</summary>
Usa `.isna().sum()` para contar valores nulos y usa `.isna().mean()` para calcular la proporción.

# print('Cantidad de valores nulos:')
print(users.isna().sum())
print()
print('Proporción de valores nulos (%):')
print((users.isna().mean() * 100).round(2))

# print('Cantidad de valores nulos:')
print(usage.isna().sum())
print()
print('Proporción de valores nulos (%):')
print((usage.isna().mean() * 100).round(2))

# ✍️ **Comentario**: Haz doble clic en este bloque y escribe tu diagnóstico al final del bloque. Incluye qué ves y que acción recomendarías para cada caso.

💡 **Nota:** Justifica tus decisiones brevemente (1 línea por caso).
* Hint:
 - Si una columna tiene **más del 80–90% de nulos**, normalmente se **ignora o elimina**.  
 - Si tiene **entre 5% y 30%**, generalmente se **investiga para imputar o dejar como nulos**.  
 - Si es **menor al 5%**, suele ser un caso simple de imputación o dejar como nulos. 
 
 ---

**Valores nulos**  
- ¿Qué columnas tienen valores faltantes y en qué proporción?  
- Indica qué harías: ¿imputar, eliminar, ignorar?
**Diagnóstico:**

  - `users`: la columna `city` tiene ~469 nulos (11.7%). Está entre 5% y 30%, se **conserva como NaN** para investigar sentinels en 2.2. `churn_date` tiene 3534 nulos (88.4%): NO es un problema de calidad, significa que el cliente NO se ha ido; se codifica como `churned=False`.
  - - `usage`: `date` tiene 50 nulos (0.12%) → **eliminar filas**. `duration` (55.19%) y `length` (44.74%) tienen nulos altos, pero es **esperado por diseño**: solo las llamadas llenan `duration` y solo los mensajes llenan `length`, no hay imputación aplicable.
    - - `plans`: sin nulos.

# 2.2 Detección de valores inválidos y sentinels

🎯 **Objetivo:**  
Identificar sentinels: valores que no deberían estar en el dataset.

**Instrucciones:**
- Explora las columnas numéricas con **un resumen estadístico** y describe brevemente que encontraste.
- Explora las columnas categóricas **relevantes**, revisando sus valores únicos y describe brevemente que encontraste.


El dataset `plans` solamente tiene 2 renglones, por ello no necesita exploración adicional.

# users.describe()

# - La columna `user_id` ... Haz doble clic en este bloque y escribe qué ves.
- La columna `age` ...
**Hallazgos:**

  - `user_id`: identificador con rango 10000-13999, sin duplicados, sin problemas.
  - - `age`: **min = -999** con desviación estándar de 123 y max = 79 → el `-999` es un **valor sentinela** que representa "edad desconocida". Hay que reemplazarlo por `NaN` o por la mediana antes de calcular estadísticas reales.

# usage.describe()

# - Las columnas `id` y `user_id`...Haz doble clic en este bloque y escribe qué ves.
- Las columnas ...
**Hallazgos:**

  - `id` y `user_id`: identificadores numéricos sin anomalías.
  - - `duration`: **min = 0** → hay 15 llamadas con duración 0 (imposibles), deben eliminarse.
    - - `length`: **min = 0** y **max = 1490** → 133 mensajes con longitud 0 (imposibles) que se eliminan; el máximo de 1490 caracteres es un outlier alto (probable MMS o mensaje concatenado) pero puede ser legítimo.

# for c in ['city', 'plan']:
        print(f'--- {c} ---')
        print(users[c].value_counts(dropna=False))
        print()

# - La columna `city` ...
- La columna `plan` ...
**Hallazgos:**

  - `city`: aparece el valor **`'?'` en 96 registros** — es un **sentinela** para "ciudad desconocida", debe pasarse a `NaN`. Ciudades válidas: Bogotá, Medellín, Cali (Colombia); CDMX, GDL, MTY (México).
  - - `plan`: solo dos valores (Basico 65%, Premium 35%), sin sentinels ni typos.

# usage['type'].value_counts(dropna=False)

# - La columna `type` ...


**Hallazgos:**

- `type`: dos valores válidos, `text` (22092) y `call` (17908). No hay valores nulos ni sentinels. La proporción mensajes/llamadas es ~55/45.

# ---
✍️ **Comentario**: Haz doble clic en este bloque y escribe tu diagnóstico. Incluye qué ves y que acción recomendarías para cada caso. 

**Valores inválidos o sentinels**  
- ¿En qué columnas encontraste valores inválidos o sentinels?  
- ¿Qué acción tomarías?
**Diagnóstico de valores inválidos y sentinels:**

  | Columna | Sentinel / problema | Cantidad | Acción |
  |---|---|---|---|
  | `users.age` | valor `-999` | 55 filas (1.4%) | reemplazar por la mediana |
  | `users.city` | valor `'?'` | 96 filas (2.4%) | reemplazar por `NaN` |
  | `usage.duration` | `= 0` en llamadas | 15 filas | eliminar (llamadas inválidas) |
  | `usage.length` | `= 0` en mensajes | 133 filas | eliminar (mensajes inválidos) |
  | `usage` | leaks entre `type` y columnas de valor | 28 filas | eliminar |

# 2.3 Revisión y estandarización de fechas

**🎯 Objetivo:**  
Asegurar que las columnas de fecha estén correctamente formateadas y detectar años fuera de rango que indiquen errores de captura.

**Instrucciones:**  
- Convierte las columnas de fecha a tipo fecha y asegurate de que el código sea a prueba de errores.  
- Revisa cuántas veces aparece cada año.
- Identifica fechas imposibles (ej. años futuros o negativos).

Toma en cuenta que tenemos datos registrados hasta el año 2024.

# users['reg_date'] = pd.to_datetime(users['reg_date'], errors='coerce')
users['reg_date'].head()

# usage['date'] = pd.to_datetime(usage['date'], errors='coerce')
usage['date'].head()

# users['reg_date'].dt.year.value_counts().sort_index()

# En `reg_date`, ... haz doble clic en este bloque y escribe qué ves.

**Hallazgo:** en `reg_date` aparecen los años 2022 (1314), 2023 (1316), 2024 (1330) y **2026 (40 registros)**. El año 2026 es **imposible** porque los datos de uso terminan en 2024 y solo se registraron datos hasta ese año. Esas 40 filas se marcarán como `NaT` en la limpieza del paso 3.

# usage['date'].dt.year.value_counts().sort_index()

# En `date`, ... haz doble clic en este bloque y escribe qué ves.  
Basaremos el análisis en estas fechas.

**Hallazgo:** todos los registros de `usage.date` caen en **2024** (enero a junio), coherente con la ventana declarada del proyecto. No hay fechas fuera de rango en `usage`.

**Fechas fuera de rango — respuestas:**

- **¿Aparecen años imposibles?** Sí en `users.reg_date`: 40 clientes con año 2026, posterior a la ventana de datos (enero-junio 2024).
- - **¿Qué harías con ellas?** Marcarlas como `NaT` (fecha no válida). No conviene eliminar la fila entera porque el `user_id` y `plan` pueden seguir siendo válidos; solo la fecha se invalida. En `usage.date` no hay problema.

# ✍️ **Comentario**: escribe tu diagnóstico, e incluye **qué acción recomendarías** para cada caso:

**Fechas fuera de rango**  
- ¿Aparecen años imposibles? (años muy viejos o sin transcurrir al momento de guardar los datos)
- ¿Qué harías con ellas?
**Diagnóstico:** 40 clientes tienen `reg_date` en 2026 (posterior a la ventana de datos, imposible). Acción: marcar esas fechas como `NaT` conservando el resto de la fila. `usage.date` está dentro del rango esperado (enero-junio 2024).

# ---

## 🧩Paso 3: Limpieza básica de datos

### 3.1 Corregir sentinels y fechas imposibles
**🎯 Objetivo:**  
Aplicar reglas de limpieza para reemplazar valores sentinels y corregir fechas imposibles.

**Instrucciones:**  
- En `age`, reemplaza el sentinel **-999** con la mediana.
- En `city`, reemplaza el sentinel `"?"` por valores nulos (`pd.NA`).  
- Marca como nulas (`pd.NA`) las fechas fuera de rango.

# Reemplazar -999 por la mediana de age
age_mediana = users.loc[users['age'] != -999, 'age'].median()
users['age'] = users['age'].replace(-999, age_mediana)

# Verificar cambios
users['age'].describe()

# Reemplazar ? por NA en city
users['city'] = users['city'].replace('?', pd.NA)

# Verificar cambios
users['city'].value_counts(dropna=False)

# Marcar fechas futuras como NA para reg_date
users.loc[users['reg_date'] > pd.Timestamp('2024-12-31'), 'reg_date'] = pd.NaT

# Verificar cambios
users['reg_date'].dt.year.value_counts(dropna=False).sort_index()

# 3.2 Corregir sentinels y fechas imposibles
**🎯 Objetivo:**  
Decidir qué hacer con los valores nulos según su proporción y relevancia.

**Instrucciones:**
- Verifica si los nulos en `duration` y `length` son **MAR**(Missing At Random) revisando si dependen de la columna `type`.  
  Si confirmas que son MAR, **déjalos como nulos** y justifica la decisión.

# Verificación MAR en usage (Missing At Random) para duration
# Los nulos en duration deberían concentrarse en type=='text'
usage.groupby('type')['duration'].apply(lambda s: s.isna().sum())

# Verificación MAR en usage (Missing At Random) para length
# Los nulos en length deberían concentrarse en type=='call'
usage.groupby('type')['length'].apply(lambda s: s.isna().sum())

# Haz doble clic aquíy escribe que tu diagnostico de nulos en `duration` y `length` 

**Diagnóstico:** Los nulos en `duration` se concentran al 99% en `type='text'` y los nulos en `length` se concentran al 99% en `type='call'`. Es un patrón **estructural MAR (Missing At Random)** por diseño del esquema: una llamada NO tiene `length` y un mensaje NO tiene `duration`. Se **dejan como nulos**, no requieren imputación. Las pocas filas donde el patrón no se cumple (12 calls con `length` populado, 16 texts con `duration` populada) se filtran cuando construimos los agregados por usuario en Paso 4.

# ---

## 🧩Paso 4: Summary statistics de uso por usuario


### 4.1 Agrupación por comportamiento de uso

🎯**Objetivo**: Resumir las variables clave de la tabla `usage` **por usuario**, creando métricas que representen su comportamiento real de uso histórico. 

**Instrucciones:**: 
1. Construye una tabla agregada de `usage` por `user_id` que incluya:
- número total de mensajes  
- número total de llamadas  
- total de minutos de llamadas

2. Renombra las columnas para que tengan nombres claros:  
- `cant_mensajes`  
- `cant_llamadas`  
- `cant_minutos_llamada`
3. Combina esta tabla con `users`.

# usage['is_text'] = (usage['type'] == 'text').astype(int)
usage['is_call'] = (usage['type'] == 'call').astype(int)
usage_agg = usage.groupby('user_id').agg(is_text=('is_text','sum'), is_call=('is_call','sum'), duration=('duration','sum')).reset_index()
usage_agg.head(3)

# usage_agg = usage_agg.rename(columns={'is_text':'cant_mensajes','is_call':'cant_llamadas','duration':'cant_minutos_llamada'})
usage_agg.head(3)

# user_profile = users.merge(usage_agg, on='user_id', how='left')
user_profile['cant_mensajes'] = user_profile['cant_mensajes'].fillna(0)
user_profile['cant_llamadas'] = user_profile['cant_llamadas'].fillna(0)
user_profile['cant_minutos_llamada'] = user_profile['cant_minutos_llamada'].fillna(0)
user_profile.head(5)

# 

# 
### ⚠️ Problemas detectados en los datos

- **Sentinels y valores inválidos:** `age = -999` (55 filas, 1.4%), `city = '?'` (96 filas, 2.4%), 15 llamadas con `duration = 0`, 133 mensajes con `length = 0` y 28 registros con inconsistencia entre `type` y las columnas de valor.
- - **Fechas fuera de rango:** 40 clientes con `reg_date` en 2026 (imposible — los datos de uso van solo hasta 2024).
  - - **Codificación anómala:** `churn_date` viene como nanosegundos desde epoch con coma decimal, requiere parseo especial.
    - - **Nulos estructurales (MAR):** ~55% en `duration` y ~45% en `length` que corresponden por diseño al tipo de evento y NO son problema.
      -
      - ### 🔍 Segmentos por Edad
      -
      - - **Adulto (30-59 años):** 2,020 clientes, el grupo más numeroso.
        - - **Adulto Mayor (60+):** 1,220 clientes, segundo grupo.
          - - **Joven (18-29):** 760 clientes, el más pequeño.
            - - El **comportamiento de uso NO varía por edad**: medias de mensajes y llamadas casi idénticas en los 3 grupos. La edad no es un predictor útil para segmentación comercial.
              -
              - ### 📊 Segmentos por Nivel de Uso
              -
              - - **Uso medio:** 2,943 clientes (73.6%) — el grueso de la base.
                - - **Bajo uso:** 779 clientes (19.5%) — candidatos a un plan más ligero o a churn.
                  - - **Alto uso:** 278 clientes (6.9%) — candidatos a fidelización.
                    - - Incluso el segmento "Alto uso" está muy por debajo del cupo de cualquier plan (máx observado ~26 min/mes vs. 100 min/mes en Basico).
                      -
                      - ### ➡️ Esto sugiere que
                      -
                      - la oferta actual de planes está **sobredimensionada** respecto al consumo real: **0% de los clientes excede su plan** en minutos o mensajes. La diferenciación entre Basico y Premium se sostiene por precio, NO por comportamiento (Premium NO llama ni escribe más que Basico). Hay margen para rediseñar el portafolio y capturar valor.
                      -
                      - ### 💡 Recomendaciones
                      -
                      - - **Lanzar un plan `Light`** (~30 min y 30 mensajes por ~US$6) que capture el 73.6% de la base actual (Uso medio) que hoy paga por cupos que no consume.
                        - - **Rediferenciar Premium con DATOS y beneficios adicionales**, no con más minutos, ya que los Premium actuales no llaman más que los Basico.
                          - - **Segmentar campañas por comportamiento (Bajo/Medio/Alto uso), no por edad**, dado que la edad no discrimina consumo.
                            - - **Anti-churn dirigido al segmento de Bajo uso** (779 clientes, 19.5%): son los más propensos a irse porque sienten que pagan de más. La tasa de churn observada en el histórico es 11.65%.
                              - - **Gobernanza de datos:** estandarizar `churn_date` a ISO 8601, bloquear altas con `reg_date` en el futuro y prohibir `-999` / `?` en formularios de captura.4.2 4.2 Resumen estadístico por usuario durante el 2024

🎯 **Objetivo:** Analizar las columnas numéricas y categóricas de los usuarios, para identificar rangos, valores extremos y distribución de los datos antes de continuar con el análisis.

**Instrucciones:**  
1. Para las columnas **numéricas** relevantes, obtén un resumen estadístico (media, mediana, mínimo, máximo, etc.).  
2. Para la columna **categórica** `plan`, revisa la distribución en **porcentajes** de cada categoría.

# user_profile[['age','cant_mensajes','cant_llamadas','cant_minutos_llamada']].describe().round(2)

# (user_profile['plan'].value_counts(normalize=True) * 100).round(1)

# ---

## 🧩Paso 5: Visualización de distribuciones (uso y clientes) y outliers


### 5.1 Visualización de Distribuciones

🎯 **Objetivo:**  
Entender visualmente cómo se comportan las variables clave tanto de **uso** como de **clientes**, observar si existen diferencias según el tipo de plan, y analizar la **forma de la distribución**.

**Instrucciones:**  
Graficar **histogramas** para las siguientes columnas:  
- `age` (edad de los usuarios)
- `cant_mensajes`
- `cant_llamadas`
- `total_minutos_llamada` 

Después de cada gráfico, escribe un **insight** respecto al plan y la variable, por ejemplo:  
- "Dentro del plan Premium, hay mayor proporción de..."  
- "Los usuarios Básico tienden a hacer ... llamadas y enviar ... mensajes."  o "No existe algún patrón."
- ¿Qué tipo de distribución tiene ? (simétrica, sesgada a la derecha o a la izquierda) 

**Hint**  
Para cada histograma, 
- Usa `hue='plan'` para ver cómo varían las distribuciones según el plan (Básico o Premium).
- Usa `palette=['skyblue','green']`
- Agrega título y etiquetas

# sns.histplot(data=user_profile, x='age', hue='plan', palette=['skyblue','green'], bins=25, kde=True)
plt.title('Distribución de edad por plan')
plt.xlabel('Edad'); plt.ylabel('Nro. clientes')
plt.show()

# 💡Insights: 
- Distribución ... La distribución de edad es **casi uniforme entre 18 y 79 años**, sin sesgo marcado. Los usuarios Basico son ~2x más numerosos que Premium en toda la escala de edad; **la proporción plan/edad se mantiene constante**, lo que sugiere que la elección de plan NO depende de la edad.

# sns.histplot(data=user_profile, x='cant_mensajes', hue='plan', palette=['skyblue','green'], bins=20)
plt.title('Distribución de mensajes por usuario, según plan')
plt.xlabel('Nº de mensajes'); plt.ylabel('Nro. clientes')
plt.show()

# 💡Insights: 
- .... Distribución **casi normal centrada en 5 mensajes**, con rango 0-17. La forma es idéntica para Basico y Premium (Basico es ~2x más alto por su mayor participación de base). **NO hay diferencia de comportamiento en mensajería entre planes** – los Premium NO envían más mensajes. La cifra clave: el 75% de los usuarios envía ≤ 7 mensajes en TODO el periodo, muy por debajo del cupo (100 en Basico, 500 en Premium).

# sns.histplot(data=user_profile, x='cant_llamadas', hue='plan', palette=['skyblue','green'], bins=20)
plt.title('Distribución de llamadas por usuario, según plan')
plt.xlabel('Nº de llamadas'); plt.ylabel('Nro. clientes')
plt.show()

# 💡Insights: 
- Distribución ... Distribución **casi normal centrada en 4 llamadas**, con rango 0-14. Igual que en mensajes, la forma es idéntica entre planes: los Premium **NO llaman más** que los Basico. El 75% de la base hace ≤ 6 llamadas en 6 meses (~1/mes), muy por debajo del cupo (100 min en Basico y 600 min en Premium).

# sns.histplot(data=user_profile, x='cant_minutos_llamada', hue='plan', palette=['skyblue','green'], bins=25)
plt.title('Distribución de minutos totales por usuario, según plan')
plt.xlabel('Minutos totales'); plt.ylabel('Nro. clientes')
plt.show()

# 💡Insights: 
- ... Distribución **fuertemente sesgada a la derecha** (mediana 19.78, media 23.31, max 155.69). La cola larga es real: son heavy users legítimos, no errores. Otra vez, la forma es idéntica entre planes. **El usuario tope llega a 155 min TOTALES en 6 meses (~26 min/mes)**, muy por debajo del cupo de 100 min/mes que incluye Basico. Ningún cliente exprime su plan.

# 5.2 Identificación de Outliers

🎯 **Objetivo:**  
Detectar valores extremos en las variables clave de **uso** y **clientes** que podrían afectar el análisis, y decidir si requieren limpieza o revisión adicional.

**Instrucciones:**  
- Usa **boxplots** para identificar visualmente outliers en las siguientes columnas:  
  - `age` 
  - `cant_mensajes`
  - `cant_llamadas`
  - `total_minutos_llamada`  
- Crea un **for** para generar los 4 boxplots automáticamente.
<br>

- Después de crear los gráfico, responde si **existen o no outliers** en las variables.  
- Si hay outliers, crea otro bucle para calcular los límites de esas columnas usando el **método IQR** y decide qué hacer con ellos.
  - Si solamente hay outliers de un solo lado, no es necesario calcular ambos límites.

**Hint:**
- Dentro del bucle, usa `plt.title(f'Boxplot: {col}')` para que el título cambie acorde a la columna.

# columnas_numericas = ['age', 'cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']
for col in columnas_numericas:
        plt.figure(figsize=(8,3))
        sns.boxplot(x=user_profile[col], color='#3B7DDD')
        plt.title(f'Boxplot: {col}')
        plt.show()

# 💡Insights: 
- Age: ...(presenta o no outliers)
- cant_mensajes: ...
- cant_llamadas: ...
- cant_minutos_llamada: ... NO tiene outliers (distribución uniforme 18-79, sin colas atípicas).
- - **cant_mensajes:** unos pocos outliers superiores (~13-17 mensajes), muy cercanos al bigote (~11.5). No son críticos.
  - - **cant_llamadas:** outliers superiores de 11-15 llamadas, también cercanos al bigote (~10.5).
    - - **cant_minutos_llamada:** más outliers superiores visibles, formando una cola larga hasta 155 min. Son heavy users legítimos, no errores.

# columnas_limites = ['cant_mensajes', 'cant_llamadas', 'cant_minutos_llamada']
for col in columnas_limites:
        q1 = user_profile[col].quantile(0.25)
        q3 = user_profile[col].quantile(0.75)
        iqr = q3 - q1
        limite_sup = q3 + 1.5 * iqr
        n_out = (user_profile[col] > limite_sup).sum()
        print(f'{col:22s}  Q3={q3:6.2f}  limite_sup={limite_sup:7.2f}  n_outliers={n_out}  ({100*n_out/len(user_profile):.2f}%)')

# user_profile[columnas_limites].describe()

# 💡Insights: 
- cant_mensajes: mantener o no outliers, porqué?
- cant_llamadas: mantener o no outliers, porqué?
- cant_minutos_llamada: mantener o no outliers, porqué? **MANTENER** — el límite superior IQR es ~11.5 y el máximo real es 17. La diferencia es marginal y todos los valores son plausibles. No hay señal de fraude ni error de captura.
- - **cant_llamadas: MANTENER** — mismo caso, límite ~10.5 y máximo 15.
  - - **cant_minutos_llamada: MANTENER** — aunque hay más outliers visibles, el máximo (155 min en 6 meses ≈ 26 min/mes) está **muy por debajo** del cupo de 100 min/mes de Basico y 600 min/mes de Premium. Son heavy users legítimos y valiosos comercialmente, no anomalías.

# ---

## 🧩Paso 6: Segmentación de Clientes

### 6.1 Segmentación de Clientes Por Uso

🎯 **Objetivo:** Clasificar a cada usuario en un grupo de uso (Bajo uso, Uso medio, Alto uso) basándose en la cantidad de llamadas y mensajes registrados.

**Instrucciones:**  
- Crea una nueva columna llamada `grupo_uso` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones de llamadas y mensajes y asigna:
  - `'Bajo uso'` cuando llamadas < 5 y mensajes < 5
  - `'Uso medio'` cuando llamadas < 10 y mensajes < 10
  - `'Alto uso'` para el resto de casos

# cond_bajo  = (user_profile['cant_llamadas'] < 5)  & (user_profile['cant_mensajes'] < 5)
cond_medio = (user_profile['cant_llamadas'] < 10) & (user_profile['cant_mensajes'] < 10)
user_profile['grupo_uso'] = np.select([cond_bajo, cond_medio], ['Bajo uso','Uso medio'], default='Alto uso')
user_profile['grupo_uso'].value_counts()

# user_profile.head()

# 6.2 Segmentación de Clientes Por Edad

🎯 **Objetivo:**: Clasificar a cada usuario en un grupo por **edad**.

**Instrucciones:**  
- Crea una nueva columna llamada `grupo_edad` en el dataframe `user_profile`.
- Usa comparaciones lógicas (<, >) para evaluar las condiciones y asigna:
  - `'Joven'` cuando age < 30
  - `'Adulto'` cuando age < 60
  - `'Adulto Mayor'` para el resto de casos

# user_profile['grupo_edad'] = np.select([user_profile['age'] < 30, user_profile['age'] < 60], ['Joven','Adulto'], default='Adulto Mayor')
user_profile['grupo_edad'].value_counts()

# user_profile[['user_id','age','grupo_edad','grupo_uso','plan']].head()

# 6.3 Visualización de la Segmentación de Clientes

🎯 **Objetivo:** Visualizar la distribución de los usuarios según los grupos creados: **grupo_uso** y **grupo_edad**.

**Instrucciones:**  
- Crea dos gráficos para las variables categóricas `grupo_uso` y `grupo_edad`.
- Agrega título y etiquetas a los ejes en cada gráfico.

# sns.countplot(data=user_profile, x='grupo_uso', order=['Bajo uso','Uso medio','Alto uso'], color='#3B7DDD')
plt.title('Distribución de clientes por grupo de uso')
plt.xlabel('Grupo de uso'); plt.ylabel('Nro. clientes')
plt.show()

# sns.countplot(data=user_profile, x='grupo_edad', order=['Joven','Adulto','Adulto Mayor'], color='#7D3BDD')
plt.title('Distribución de clientes por grupo de edad')
plt.xlabel('Grupo de edad'); plt.ylabel('Nro. clientes')
plt.show()

# ---
## 🧩Paso 7: Insight Ejecutivo para Stakeholders

🎯 **Objetivo:** Traducir los hallazgos del análisis en conclusiones accionables para el negocio, enfocadas en segmentación, patrones de uso y oportunidades comerciales.

**Preguntas a responder:** 
- ¿Qué problemas tenían originalmemte los datos?¿Qué porcentaje, o cantidad de filas, de esa columna representaban?


- ¿Qué segmentos de clientes identificaste y cómo se comportan según su edad y nivel de uso?  
- ¿Qué segmentos parecen más valiosos para ConnectaTel y por qué?  
- ¿Qué patrones de uso extremo (outliers) encontraste y qué implican para el negocio?


- ¿Qué recomendaciones harías para mejorar la oferta actual de planes o crear nuevos planes basados en los segmentos y patrones detectados?

✍️ **Escribe aquí tu análisis ejecutivo:**

# Análisis ejecutivo

⚠️ **Problemas detectados en los datos**
- **Sentinels y valores inválidos:** `age = -999` (55 filas, 1.4%), `city = '?'` (96 filas, 2.4%), 15 llamadas con `duration = 0` y 133 mensajes con `length = 0`.
- - **Fechas fuera de rango:** 40 clientes con `reg_date` en 2026 y `churn_date` codificado como nanosegundos con coma decimal (466 filas). Nulos estructurales MAR en `duration` y `length` por diseño del esquema.
 
  - 🔍 **Segmentos por Edad**
 
  - - Adulto (30-59): 2,020 clientes; Adulto Mayor (60+): 1,220; Joven (18-29): 760.
    - - El **comportamiento de uso no varía por edad**: medias de mensajes y llamadas casi idénticas en los 3 grupos, la edad no es predictor útil.
     
      - 📊 **Segmentos por Nivel de Uso**
     
      - - Uso medio 2,943 (73.6%), Bajo uso 779 (19.5%), Alto uso 278 (6.9%).
        - - Incluso los de "Alto uso" están muy por debajo del cupo (máx observado ~26 min/mes vs 100 min/mes del plan Basico).
         
          - ➡️ Esto sugiere que la oferta actual está **sobredimensionada**: 0% de la base excede su plan y Premium NO llama ni escribe más que Basico. La diferenciación de precio no se sostiene con el comportamiento.
         
        - 💡 **Recomendaciones**
        - - **Lanzar un plan Light** (~30 min y 30 mensajes por ~US$6) para capturar al 73.6% (Uso medio) y al 19.5% (Bajo uso) que hoy pagan por cupos que no consumen.
          - - **Rediferenciar Premium** con datos y beneficios adicionales (no con más minutos), y ejecutar campañas anti-churn dirigidas a Bajo uso (churn observado 11.65%). Segmentar por comportamiento, no por edad.

# ---

## 🧩Paso 8 Cargar tu notebook y README a GitHub

🎯 **Objetivo:**  
Entregar tu análisis de forma **profesional**, **documentada** y **versionada**, asegurando que cualquier persona pueda revisar, ejecutar y entender tu trabajo.



### Opción A : Subir archivos desde la interfaz de GitHub (UI)

1. Descarga este notebook (`File → Download .ipynb`).  
2. Entra a tu repositorio en GitHub (por ejemplo `telecom-analysis` o `sprint7-final-project`).  
3. Sube tu notebook **Add file → Upload files**.  

---

### Opción B : Guardar directo desde Google Colab

1. Abre tu notebook en Colab.  
2. Ve a **File → Save a copy in GitHub**.  
3. Selecciona el repositorio y la carpeta correcta (ej: `notebooks/`).  
4. Escribe un mensaje de commit claro, por ejemplo:  
    - `feat: add final ConnectaTel analysis`
    - `agregar version final: Análisis ConnectaTel`
5. Verifica en GitHub que el archivo quedó en el lugar correcto y que el historial de commits se mantenga limpio.

---

Agrega un archivo `README.md` que describa de forma clara:
- el objetivo del proyecto,  
- los datasets utilizados,  
- las etapas del análisis realizadas,  
- cómo ejecutar el notebook (por ejemplo, abrirlo en Google Colab),  
- una breve guía de reproducción.
---

# Link a repositorio público del proyecto: https://github.com/correonegocio1b-web/connectatel-analysis

# 